In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, math, re, time, sys, subprocess, pkgutil, traceback, html
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from peft import PeftModel
from transformers import AutoTokenizer, AutoConfig, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

if pkgutil.find_loader("gradio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=4.44.0"])
import gradio as gr

In [ ]:
MODEL_DIR = "/content/drive/MyDrive/B8_light_best_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
with open(os.path.join(MODEL_DIR, "export_meta.json"), "r", encoding="utf-8") as f:
    META = json.load(f)

MODEL_NAME = META["model_name"]
MAX_LENGTH = META.get("max_length", 1536)
HEAD_DROPOUT = META.get("head_dropout", 0.1)

TR_FEATURE_COLS = META["tr_feature_cols"]
CC_FEATURE_COLS = META["cc_feature_cols"]
LR_FEATURE_COLS = META["lr_feature_cols"]
GRA_FEATURE_COLS = META["gra_feature_cols"]

tr_feat_mean, tr_feat_std = np.array(META["tr_feat_mean"], dtype=np.float32), np.array(META["tr_feat_std"], dtype=np.float32)
cc_feat_mean, cc_feat_std = np.array(META["cc_feat_mean"], dtype=np.float32), np.array(META["cc_feat_std"], dtype=np.float32)
lr_feat_mean, lr_feat_std = np.array(META["lr_feat_mean"], dtype=np.float32), np.array(META["lr_feat_std"], dtype=np.float32)
gra_feat_mean, gra_feat_std = np.array(META["gra_feat_mean"], dtype=np.float32), np.array(META["gra_feat_std"], dtype=np.float32)

In [ ]:
STOPWORDS = {"a","an","the","and","or","but","if","while","is","am","are","was","were","be","been","being","of","to","in","on","for","with","as","at","by","from","that","this","these","those","it","its","he","she","they","them","their","we","our","you","your","i","me","my","mine","his","her","hers","do","does","did","have","has","had","will","would","can","could","should","may","might","not","so","than","then","there","here","about","into","over","after","before","more","most","some","any","such","no","nor","too","very"}
DISCOURSE_MARKERS = ["however", "therefore", "moreover", "furthermore", "in addition", "for example", "for instance", "on the one hand", "on the other hand", "in conclusion", "to conclude", "in summary", "as a result", "firstly", "secondly", "finally", "besides", "nevertheless", "thus", "overall", "in contrast", "for this reason"]

def safe_text(x): 
    return str(x).strip() if x is not None and not (isinstance(x, float) and np.isnan(x)) else ""
def normalize_text(text): 
    return re.sub(r"\s+", " ", safe_text(text).lower()).strip()
def tokenize_words(text): 
    return re.findall(r"[a-zA-Z']+", normalize_text(text))
def split_sentences(text): 
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", safe_text(text).strip()) if s.strip()]
def split_paragraphs(text): 
    paras = [p.strip() for p in re.split(r"\n\s*\n+", safe_text(text)) if p.strip()]; return paras if paras else [safe_text(text).strip()]
def word_count(text): 
    return len(tokenize_words(text))

def extract_tr_features(prompt, essay):
    pw = set([w for w in tokenize_words(prompt) if w not in STOPWORDS and len(w)>2])
    ew = set(tokenize_words(essay))
    cov = len(pw & ew) / len(pw) if pw else 0.0
    sim = len(pw & set([w for w in ew if w not in STOPWORDS and len(w)>2])) / math.sqrt(len(pw)*len(ew)) if pw and ew else 0.0
    low = normalize_text(essay)
    return {
        "tr_prompt_essay_sim": float(sim), "tr_prompt_keyword_coverage": float(cov),
        "tr_has_opinion": float(any(p in low for p in ["i believe", "i think", "in my opinion", "personally"])),
        "tr_has_both_views": float("on the one hand" in low and "on the other hand" in low),
        "tr_has_example": float(any(p in low for p in ["for example", "for instance", "such as"])),
        "tr_has_conclusion": float(any(p in low for p in ["in conclusion", "to conclude", "to sum up", "overall"])),
        "tr_word_count": float(word_count(essay)),
    }

def extract_cc_features(prompt, essay):
    paras, sents = split_paragraphs(essay), split_sentences(essay)
    sent_lens = [len(tokenize_words(s)) for s in sents if len(tokenize_words(s))>0]
    low = normalize_text(essay)
    dm_count = sum(low.count(m) for m in DISCOURSE_MARKERS)
    dm_div = sum(1 for m in DISCOURSE_MARKERS if low.count(m) > 0)
    return {
        "cc_num_paragraphs": float(len(paras)), "cc_avg_paragraph_len": float(np.mean([word_count(p) for p in paras])) if paras else 0.0,
        "cc_avg_sentence_len": float(np.mean(sent_lens)) if sent_lens else 0.0, "cc_sentence_len_std": float(np.std(sent_lens)) if sent_lens else 0.0,
        "cc_discourse_marker_count": float(dm_count), "cc_discourse_marker_diversity": float(dm_div),
    }

def extract_lr_features(prompt, essay):
    words = tokenize_words(essay)
    return {
        "lr_root_ttr": float(len(set(words))/math.sqrt(len(words))) if words else 0.0,
        "lr_avg_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
        "lr_long_word_ratio": float(sum(len(w)>=7 for w in words)/len(words)) if words else 0.0,
        "lr_repetition_ratio": float(sum(v for v in Counter(words).values() if v>1)/len(words)) if len(words)>1 else 0.0,
        "lr_unique_word_ratio": float(len(set(words))/len(words)) if words else 0.0,
        "lr_lexical_density_proxy": float(len([w for w in words if len(w)>3 and w not in STOPWORDS])/len(words)) if words else 0.0,
    }

def extract_gra_features(prompt, essay):
    words, sents = tokenize_words(essay), split_sentences(essay)
    sent_lens = [len(tokenize_words(s)) for s in sents if len(tokenize_words(s))>0]
    return {
        "gf_word_count": float(len(words)), "gf_sentence_count": float(len(sent_lens)),
        "gf_avg_sentence_len": float(np.mean(sent_lens)) if sent_lens else 0.0,
        "gf_short_sentence_ratio": float(sum(l<8 for l in sent_lens)/len(sent_lens)) if sent_lens else 0.0,
        "gf_long_sentence_ratio": float(sum(l>30 for l in sent_lens)/len(sent_lens)) if sent_lens else 0.0,
        "gf_punct_density": float(len(re.findall(r"[.!?,;:]", essay))/max(len(words),1)),
        "gf_repeated_punct_ratio": float(len(re.findall(r"([!?.,;:])\1+", essay))/max(len(essay),1)),
        "gf_lowercase_sent_start_ratio": 0.0, "gf_lowercase_i_ratio": 0.0,
        "gf_repeated_word_ratio": float(sum(1 for i in range(1, len(words)) if words[i]==words[i-1])/len(words)) if len(words)>1 else 0.0,
        "gf_missing_terminal_punct": float(safe_text(essay).strip()[-1] not in ".!?") if safe_text(essay).strip() else 1.0,
    }

def build_input_text(prompt, essay): 
    return f"You are an IELTS Writing Task 2 examiner\nTASK PROMPT:\n{safe_text(prompt)}\n\nESSAY:\n{safe_text(essay)}"
def standardize_features(feat_dict, cols, mean_, std_): 
    return (np.array([feat_dict[c] for c in cols], dtype=np.float32) - mean_) / np.where(std_<1e-6, 1.0, std_)
def round_to_half(x): 
    return float(np.floor(np.asarray(x, dtype=np.float32) * 2 + 0.5) / 2)

In [ ]:
class IELTSBranch(nn.Module):
    def __init__(self, hidden_size, feat_dim, fusion_dim=256, head_dropout=0.1):
        super().__init__()
        self.llm_proj = nn.Sequential(nn.Linear(hidden_size, fusion_dim))
        self.feat_proj = nn.Sequential(nn.Linear(feat_dim, fusion_dim))
        self.gate = nn.Sequential(nn.Linear(hidden_size + feat_dim, fusion_dim), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(fusion_dim, fusion_dim))
        self.out = nn.Sequential(nn.Linear(fusion_dim, fusion_dim // 2), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(fusion_dim // 2, 1))
        
    def forward(self, llm_emb, feat_emb):
        l_proj, f_proj = self.llm_proj(llm_emb), self.feat_proj(feat_emb)
        g = torch.sigmoid(self.gate(torch.cat([llm_emb, feat_emb], dim=-1)))
        return self.out(g * l_proj + (1.0 - g) * f_proj), g.mean(dim=-1, keepdim=True)

class QwenForIELTSMultiTask(nn.Module):
    def __init__(self, model_name: str, tokenizer, head_dropout: float = 0.1, fusion_dim=256):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.pad_token_id = tokenizer.pad_token_id
        base_model = AutoModel.from_pretrained(model_name, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
        base_model.config.pad_token_id = tokenizer.pad_token_id
        base_model.config.use_cache = False
        self.backbone = PeftModel.from_pretrained(base_model, MODEL_DIR)
        
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(head_dropout)
        
        self.tr_branch = IELTSBranch(hidden_size, len(TR_FEATURE_COLS), fusion_dim, head_dropout)
        self.cc_branch = IELTSBranch(hidden_size, len(CC_FEATURE_COLS), fusion_dim, head_dropout)
        self.lr_branch = IELTSBranch(hidden_size, len(LR_FEATURE_COLS), fusion_dim, head_dropout)
        self.gra_branch = IELTSBranch(hidden_size, len(GRA_FEATURE_COLS), fusion_dim, head_dropout)

    def _mean_pool(self, hidden_states, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids=None, attention_mask=None, tr_features=None, cc_features=None, lr_features=None, gra_features=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        pooled = self.dropout(self._mean_pool(outputs.last_hidden_state, attention_mask))
        head_dtype = self.tr_branch.llm_proj[0].weight.dtype
        pooled = pooled.to(dtype=head_dtype)

        tr, tr_gate = self.tr_branch(pooled, tr_features.to(device=pooled.device, dtype=head_dtype))
        cc, cc_gate = self.cc_branch(pooled, cc_features.to(device=pooled.device, dtype=head_dtype))
        lr, lr_gate = self.lr_branch(pooled, lr_features.to(device=pooled.device, dtype=head_dtype))
        gra, gra_gate = self.gra_branch(pooled, gra_features.to(device=pooled.device, dtype=head_dtype))

        return {"logits": torch.cat([tr, cc, lr, gra], dim=1), "tr_gate": tr_gate, "cc_gate": cc_gate, "lr_gate": lr_gate, "gra_gate": gra_gate}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = QwenForIELTSMultiTask(MODEL_NAME, tokenizer, head_dropout=HEAD_DROPOUT)
head_state = torch.load(os.path.join(MODEL_DIR, "custom_heads.pt"), map_location="cpu")
model.tr_branch.load_state_dict(head_state["tr_branch"])
model.cc_branch.load_state_dict(head_state["cc_branch"])
model.lr_branch.load_state_dict(head_state["lr_branch"])
model.gra_branch.load_state_dict(head_state["gra_branch"])
model.to(DEVICE)
model.eval()

In [ ]:
@torch.no_grad()
def predict_ielts(prompt: str, essay: str):
    tr_feat, cc_feat, lr_feat, gra_feat = extract_tr_features(prompt, essay), extract_cc_features(prompt, essay), extract_lr_features(prompt, essay), extract_gra_features(prompt, essay)
    tr_arr = standardize_features(tr_feat, TR_FEATURE_COLS, tr_feat_mean, tr_feat_std)
    cc_arr = standardize_features(cc_feat, CC_FEATURE_COLS, cc_feat_mean, cc_feat_std)
    lr_arr = standardize_features(lr_feat, LR_FEATURE_COLS, lr_feat_mean, lr_feat_std)
    gra_arr = standardize_features(gra_feat, GRA_FEATURE_COLS, gra_feat_mean, gra_feat_std)

    enc = tokenizer(build_input_text(prompt, essay), truncation=True, max_length=MAX_LENGTH, padding=False, return_tensors="pt")
    input_ids, attention_mask = enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                    tr_features=torch.tensor(tr_arr).unsqueeze(0).to(DEVICE), cc_features=torch.tensor(cc_arr).unsqueeze(0).to(DEVICE),
                    lr_features=torch.tensor(lr_arr).unsqueeze(0).to(DEVICE), gra_features=torch.tensor(gra_arr).unsqueeze(0).to(DEVICE))

    preds = np.clip(outputs["logits"].squeeze(0).detach().float().cpu().numpy(), 4.0, 9.0)
    raw_scores = {"TR": float(preds[0]), "CC": float(preds[1]), "LR": float(preds[2]), "GRA": float(preds[3])}
    final_scores = {k: round_to_half(v) for k, v in raw_scores.items()}
    final_scores["Overall"] = round_to_half(np.mean(list(final_scores.values())))

    return {"raw_scores": raw_scores, "final_scores": final_scores, "features": {"tr": tr_feat, "cc": cc_feat, "lr": lr_feat, "gra": gra_feat}}

def get_pred_scores(prompt, essay):
    res = predict_ielts(prompt, essay)
    return res, res["final_scores"]

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
def embed_texts(texts, batch_size=32): return embedding_model.encode(texts, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True)

In [ ]:
def build_doc_text(row): 
    return f"[PROMPT]\n{safe_text(row['prompt'])}\n\n[ESSAY]\n{safe_text(row['essay'])}".strip()
def build_retrieval_database(csv_path):
    df = pd.read_csv(csv_path)
    df["doc_text"] = df.apply(build_doc_text, axis=1)
    return df, embed_texts(df["doc_text"].tolist())

In [ ]:
EXPLAIN_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
explain_tokenizer = AutoTokenizer.from_pretrained(EXPLAIN_MODEL_NAME, use_fast=False)
if explain_tokenizer.pad_token is None: explain_tokenizer.pad_token = explain_tokenizer.eos_token
explain_tokenizer.padding_side = "left"
explain_model = AutoModelForCausalLM.from_pretrained(EXPLAIN_MODEL_NAME, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, device_map="auto").eval()

In [ ]:
def retrieve_similar_essays_for_inference(prompt, essay, df, db_embeddings, top_k_vector=20, top_k_final=5, pred_result=None, pred_scores=None):
    if pred_result is None or pred_scores is None: pred_result, pred_scores = get_pred_scores(prompt, essay)
    q_text = f"IELTS Writing Task 2 Prompt:\n{prompt}\n\nEssay:\n{essay}\n\nPredicted scores: TR={pred_scores['TR']}, CC={pred_scores['CC']}, LR={pred_scores['LR']}, GRA={pred_scores['GRA']}"
    sims = cosine_similarity(embed_texts([q_text])[0].reshape(1, -1), db_embeddings)[0]
    candidate_idx = np.argsort(-sims)[:top_k_vector]
    candidates = df.iloc[candidate_idx].copy()
    candidates["final_score"] = sims[candidate_idx]
    return {"pred_scores": pred_scores, "top_cases": candidates.sort_values("final_score", ascending=False).head(top_k_final)}

In [ ]:
def mistral_explain_writer(prompt_text, temperature=0.15, max_new_tokens=600):
    full_prompt = f"[INST]\nYou are an expert IELTS examiner.\n{prompt_text}\n[/INST]"
    model_inputs = explain_tokenizer([full_prompt], return_tensors="pt", truncation=True, max_length=4096).to(explain_model.device)
    gen_kwargs = {"max_new_tokens": max_new_tokens, "repetition_penalty": 1.05}
    if temperature > 0: gen_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": 0.9})
    with torch.no_grad():
        output_ids = explain_model.generate(**model_inputs, **gen_kwargs)
    return explain_tokenizer.batch_decode(output_ids[:, model_inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

def safe_json_loads(text, default=None):
    if default is None: default = {}
    match = re.search(r"\{.*\}", str(text).strip(), re.DOTALL)
    try: return json.loads(match.group(0)) if match else default
    except Exception: return default

In [ ]:
def tool_predict_scores(prompt, essay, state):
    res, scores = get_pred_scores(prompt, essay)
    state["pred_scores"] = scores
    return {"scores": scores}

In [ ]:
def tool_detect_task_mismatch(prompt, essay, state):
    sys_prompt = f"Evaluate if the essay is off-topic. Prompt: {prompt}\nEssay: {essay}\nOutput JSON with 'verdict' (pass/fail), 'reason'."
    resp = mistral_explain_writer(sys_prompt, temperature=0.1, max_new_tokens=200)
    parsed = safe_json_loads(resp, {"verdict": "pass", "reason": "Default pass"})
    state["task_check"] = parsed
    return parsed

In [ ]:
def tool_retrieve_similar_essays(prompt, essay, df, db_embeddings, state, top_k_vector=20, top_k_final=5):
    res = retrieve_similar_essays_for_inference(prompt, essay, df, db_embeddings, top_k_vector, top_k_final, pred_scores=state.get("pred_scores"))
    state["top_cases"] = res["top_cases"]
    return {"status": f"Retrieved {len(res['top_cases'])} cases"}

In [ ]:
def tool_generate_feedback(prompt, essay, state):
    scores = state.get("pred_scores", {"TR": 6, "CC": 6, "LR": 6, "GRA": 6})
    req = f"Write feedback for IELTS essay. Scores: {scores}.\nEssay: {essay}\nOutput JSON format: {{'TR': {{'Strength':'', 'Limitation':'', 'Revision':''}}, 'CC':...}}"
    resp = mistral_explain_writer(req, temperature=0.2, max_new_tokens=800)
    parsed = safe_json_loads(resp, {"TR": {"Strength": "", "Limitation": "", "Revision": ""}, "CC": {}, "LR": {}, "GRA": {}})
    state["feedback"] = parsed
    return {"status": "Feedback generated."}

In [ ]:
def tool_verify_feedback(prompt, essay, state):
    fb = state.get("feedback", {})
    req = f"Verify this feedback to ensure it matches the essay.\nEssay: {essay}\nFeedback: {fb}\nOutput JSON with 'verdict' (pass/revise) and 'issues' list."
    resp = mistral_explain_writer(req, temperature=0.1, max_new_tokens=300)
    parsed = safe_json_loads(resp, {"verdict": "pass", "issues": []})
    state["verification"] = parsed
    return parsed

In [ ]:
def tool_revise_feedback(prompt, essay, state):
    fb = state.get("feedback", {})
    issues = state.get("verification", {}).get("issues", [])
    req = f"Revise this feedback to fix the issues: {issues}\nOld Feedback: {fb}\nOutput JSON same format as original feedback."
    resp = mistral_explain_writer(req, temperature=0.2, max_new_tokens=800)
    parsed = safe_json_loads(resp, fb)
    state["feedback"] = parsed
    state["revision_count"] = state.get("revision_count", 0) + 1
    return {"status": "Feedback revised."}

In [ ]:
TOOL_DEFINITIONS = [
    {"name": "predict_scores", "description": "Predict IELTS scores"},
    {"name": "detect_task_mismatch", "description": "Check if essay is off-topic"},
    {"name": "retrieve_similar_essays", "description": "Find similar essays from DB"},
    {"name": "generate_feedback", "description": "Generate criteria feedback"},
    {"name": "verify_feedback", "description": "Verify feedback accuracy"},
    {"name": "revise_feedback", "description": "Revise feedback if verify fails"}
]

def choose_next_tool(prompt, essay, state, verbose=False):
    # Logic tự động hóa tiến trình
    if state["pred_scores"] is None: 
        return {"chosen_tool": "predict_scores", "arguments": {}, "source": "rule", "thought": "Need scores first", "fallback_reason": "", "raw_decision": "", "parsed_tool_call": {}}
    if state["task_check"] is None: 
        return {"chosen_tool": "detect_task_mismatch", "arguments": {}, "source": "rule", "thought": "Checking task mismatch", "fallback_reason": "", "raw_decision": "", "parsed_tool_call": {}}
    if state["top_cases"] is None: 
        return {"chosen_tool": "retrieve_similar_essays", "arguments": {}, "source": "rule", "thought": "Retrieving context", "fallback_reason": "", "raw_decision": "", "parsed_tool_call": {}}
    if state["feedback"] is None: 
        return {"chosen_tool": "generate_feedback", "arguments": {}, "source": "rule", "thought": "Generating feedback", "fallback_reason": "", "raw_decision": "", "parsed_tool_call": {}}
    if state["verification"] is None: 
        return {"chosen_tool": "verify_feedback", "arguments": {}, "source": "rule", "thought": "Verifying feedback", "fallback_reason": "", "raw_decision": "", "parsed_tool_call": {}}
    
    verdict = str(state["verification"].get("verdict", "")).lower()
    if verdict == "revise" and state.get("revision_count", 0) < 2: 
        return {"chosen_tool": "revise_feedback", "arguments": {}, "source": "rule", "thought": "Revising feedback", "fallback_reason": "", "raw_decision": "", "parsed_tool_call": {}}
    
    return {"chosen_tool": None, "arguments": {}, "source": "rule", "thought": "Pipeline complete", "fallback_reason": "", "raw_decision": "", "parsed_tool_call": {}}

def execute_tool_call(tool_name, arguments, prompt, essay, df, db_embeddings, state, top_k_vector=20, top_k_final=5):
    if tool_name == "predict_scores": 
        return tool_predict_scores(prompt, essay, state)
    if tool_name == "detect_task_mismatch": 
        return tool_detect_task_mismatch(prompt, essay, state)
    if tool_name == "retrieve_similar_essays": 
        return tool_retrieve_similar_essays(prompt, essay, df, db_embeddings, state, top_k_vector, top_k_final)
    if tool_name == "generate_feedback": 
        return tool_generate_feedback(prompt, essay, state)
    if tool_name == "verify_feedback": 
        return tool_verify_feedback(prompt, essay, state)
    if tool_name == "revise_feedback": 
        return tool_revise_feedback(prompt, essay, state)
    return {"error": "Unknown tool"}

In [ ]:
CRITERIA = ["TR", "CC", "LR", "GRA"]
def run_agent_feedback_pipeline(prompt, essay, df, db_embeddings, max_steps=8, top_k_vector=20, top_k_final=5, verbose=True):
    state = {"pred_scores": None, "task_check": None, "top_cases": None, "feedback": None, "verification": None, "trace": [], "revision_count": 0, "done": False, "done_reason": None}

    for step in range(max_steps):
        decision = choose_next_tool(prompt, essay, state, verbose=verbose)
        tool_name, thought = decision["chosen_tool"], decision["thought"]

        if tool_name is None:
            state["done_reason"] = "All steps completed."
            break

        state["trace"].append({"step": step + 1, "chosen_tool": tool_name, "thought": thought})
        tool_result = execute_tool_call(tool_name, {}, prompt, essay, df, db_embeddings, state, top_k_vector, top_k_final)
        state["trace"][-1]["tool_result"] = tool_result

        if tool_name == "verify_feedback" and state["verification"]:
            verdict = str(state["verification"].get("verdict", "")).strip().lower()
            if verdict == "pass" or (verdict == "revise" and state["revision_count"] >= 1):
                state["done"] = True
                state["done_reason"] = "Feedback verified successfully."
                break

    if not state["done"] and not state["done_reason"]: state["done_reason"] = "Stopped by max_steps."
    return state

In [ ]:
TRAIN_CSV = "/content/drive/MyDrive/ielts_train_aug_df.csv" 
df_retrieval, db_embeddings = build_retrieval_database(TRAIN_CSV)

In [ ]:
default_prompt = "Some people think that advertisements aimed at children should be banned. To what extent do you agree or disagree?"
default_essay = "It is true that social media platforms such as Facebook and Twitter have become highly popular... (Sample essay text here) In conclusion, I believe the disadvantages are more significant."

def run_gradio_agent_demo(prompt, essay, max_steps, top_k_vector, top_k_final):
    start = time.time()
    try:
        state = run_agent_feedback_pipeline(prompt, essay, df_retrieval, db_embeddings, int(max_steps), int(top_k_vector), int(top_k_final), verbose=False)
        elapsed = time.time() - start
        
        # Format Text Outputs
        summary = f"**Chạy xong trong {elapsed:.2f}s.**\nTrạng thái: {state.get('done_reason')}"
        scores_md = "\n".join([f"- **{k}**: {v}" for k,v in state.get("pred_scores", {}).items()]) if state.get("pred_scores") else "N/A"
        
        fb = state.get("feedback", {})
        fb_md = "\n\n".join([f"### {c}\n**Strength**: {fb.get(c,{}).get('Strength','')}\n**Limitation**: {fb.get(c,{}).get('Limitation','')}\n**Revision**: {fb.get(c,{}).get('Revision','')}" for c in CRITERIA]) if fb else "N/A"
        
        cases_df = state.get("top_cases", pd.DataFrame())
        if not cases_df.empty: cases_df = cases_df[["band", "TR", "CC", "LR", "GRA", "prompt", "essay"]]
        
        trace_html = "<ul>" + "".join([f"<li><b>Step {t['step']}: {t['chosen_tool']}</b> - {t['thought']}</li>" for t in state.get("trace", [])]) + "</ul>"
        
        return summary, scores_md, fb_md, cases_df, trace_html, state
    except Exception as e:
        return f"Lỗi: {str(e)}", "N/A", "N/A", pd.DataFrame(), "", {}

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# IELTS AES + Agentic RAG Pipeline (Version 8)")
    with gr.Row():
        with gr.Column():
            prompt_in = gr.Textbox(label="Prompt", value=default_prompt)
            essay_in = gr.Textbox(label="Essay", lines=15, value=default_essay)
            run_btn = gr.Button("Chấm Điểm & Nhận Xét", variant="primary")
        with gr.Column():
            summary_out = gr.Markdown()
            with gr.Tabs():
                with gr.Tab("Scores"): scores_out = gr.Markdown()
                with gr.Tab("Feedback"): feedback_out = gr.Markdown()
                with gr.Tab("Retrieved Cases"): cases_df_out = gr.Dataframe()
                with gr.Tab("Agent Trace Log"): trace_out = gr.HTML()
                with gr.Tab("Raw JSON State"): state_out = gr.JSON()
                
    run_btn.click(run_gradio_agent_demo, inputs=[prompt_in, essay_in, gr.State(8), gr.State(20), gr.State(5)], outputs=[summary_out, scores_out, feedback_out, cases_df_out, trace_out, state_out])

demo.queue().launch(debug=True, share=True)